# math-eval quickstart

```text
canonical JSONL -> evaluate.py -> raw -> replay_evaluation.py -> parsed + metrics
```

生成与评测刻意分开：`evaluate.py` 只生成可续传 raw；`replay_evaluation.py` 再依次运行 parser 和 metrics。这样更换 parser 不需要重新调用模型。canonical 每行只需 `id`、`problem`、`answer`。

## 1. 安装

本地 vLLM 使用完整的固定 GPU 环境：

```bash
./install.sh .venv
```

只调用 OpenAI-compatible API 不需要 vLLM、PyTorch 或 CUDA。下面的轻量环境同时包含生成所需的 PyYAML，以及 replay 所需的 Math-Verify：

```bash
uv python install 3.12.11
uv venv .venv-api --python 3.12.11
uv pip install --python .venv-api/bin/python pyyaml==6.0.3 math-verify==0.9.0
```

## 2. 本地 vLLM

`configs/qwen25_3b_instruct.yaml` 使用公开 model ID 和仓库 10 题，是可直接修改的 TP1 示例：

```bash
.venv/bin/python scripts/evaluate.py \
  --config configs/qwen25_3b_instruct.yaml \
  --run-id quickstart-vllm
```

中断后原命令追加 `--resume`。其他 Qwen/Llama 示例也在 `configs/`，`model.path` 可改为本地 checkpoint。模型、GPU 数量和采样参数在 YAML 修改；必填 `prompt` 指向 `prompts/` 下的 UTF-8 文件或 chat 消息目录，正文中的 `{{problem}}` 恰好替换一次。全部字段见 `configs/reference_vllm.yaml`。

## 3. OpenAI-compatible API

`configs/deepseek_api.yaml` 已配置 DeepSeek API，并限制为第一题。运行下面单元后输入 key；key 只保留在本次调用的内存中，不写入 notebook、配置或产物。其他 provider 可复制 `configs/reference_openai.yaml`。

In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os, subprocess

PYTHON = '.venv-api/bin/python'
RUN_ID = f'quickstart-api-{datetime.now():%Y%m%d-%H%M%S}'
RUN_DIR = Path('outputs/runs') / RUN_ID

def run_api():
    env = os.environ.copy()
    for name in ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'http_proxy', 'https_proxy', 'all_proxy'):
        env.pop(name, None)
    env['NO_PROXY'] = env['no_proxy'] = '*'
    env['DEEPSEEK_API_KEY'] = getpass('DEEPSEEK_API_KEY: ')
    subprocess.run(
        [PYTHON, 'scripts/evaluate.py', '--config', 'configs/deepseek_api.yaml', '--run-id', RUN_ID],
        check=True, env=env,
    )

run_api()
del run_api

In [ ]:
import json

raw_path = next((RUN_DIR / 'raw').glob('**/*.jsonl'))
raw_row = json.loads(raw_path.read_text(encoding='utf-8').splitlines()[0])
{key: raw_row.get(key) for key in ('problem', 'gold_answer', 'final_text', 'finish_reason')}

## 4. Parser + metrics

官方入口是一条 replay 命令：先把每条 `final_text` 与 `gold_answer` 交给默认 `math-v5-dual` parser，再分别输出 strict 正式分数与 soft 诊断指标。

```bash
.venv-api/bin/python scripts/replay_evaluation.py \
  --run-dir outputs/runs/<run-id> --k 1
```

下面直接评测刚才的 API run，并显示答案判断与核心指标。若评测 vLLM run，把 `PYTHON` 和 `RUN_DIR` 改为 `.venv/bin/python`、`outputs/runs/quickstart-vllm`。

In [ ]:
subprocess.run(
    [PYTHON, 'scripts/replay_evaluation.py', '--run-dir', str(RUN_DIR), '--k', '1'],
    check=True,
)

parsed_path = next((RUN_DIR / 'parsed').glob('*/parsed.jsonl'))
metrics_path = next((RUN_DIR / 'metrics').glob('*/metrics.json'))
parsed_row = json.loads(parsed_path.read_text(encoding='utf-8').splitlines()[0])
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
{
    'strict': {key: parsed_row.get(key) for key in ('status', 'is_correct', 'candidate_text')},
    'soft': {key: parsed_row['soft'].get(key) for key in ('status', 'is_correct', 'candidate_text')},
    'metrics': {
        'sample_count': metrics['sample_count'],
        'strict_accuracy': metrics['strict']['accuracy'],
        'soft_accuracy': metrics['soft']['accuracy'],
        'soft_recovery_count': metrics['soft_recovery_count'],
        'k': metrics['strict']['k'],
    },
}